# Setup lncRNA-TF pairs

### Author: Martin Loza
### Date: 25/12/09

After selecting the window and genes of interest, we will setup gene pairs for co-expression analysis with ENCODE and GTEx data

In [ ]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(biomaRt)
})

# Local variables 
seed = 777
date = "251215"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

ensembl_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"
in_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/"

# Local Functions


### Load and setup the data
#### Select lncRNA-TF gene pairs within 10Kb distance

In [2]:
# Load the setup transcripts data
# We have different species, so let's create a list to store the data
data_list = list()

# Search for the available files
files <- list.files(in_dir)

# Load the data for each species
for (file in files) {
    # Remove the underscore and everything after it to get the species names
    species_name <- str_replace(file, "_.*", "")
    data_list[[species_name]] <- read.table(file.path(in_dir, file), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)
}

head(data_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE


In [3]:
# Let's place an order in the plots
ordered_species <- c("human", "mouse")
# Arrange the data_list according to the ordered_species
data_selected_list <- data_list[ordered_species]

Let's also focus only in standard chromosomes

In [4]:
unique(data_selected_list[["mouse"]]$chromosome)

[1] "6"  "2"  "10" "11" "12" "13" "14" "15" "18" "1"  "3"  "5"  "7"  "X"  "8" 
[16] "9"  "16" "17" "4"  "19" "Y"

In [5]:
# remove data_list to avoid confusion
rm(data_list)

Let's focus only in lncRNAs

In [6]:
# Select only observations related to lncRNA
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        filter(gene_biotype == "lncRNA")
}

In [7]:
head(data_selected_list[["human"]], 3)
table(data_selected_list[["human"]]$gene_biotype)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE



 lncRNA 
4155667 

Now, let's select only those PCGs annotated as TF

In [8]:
data_selected_list[["human"]]<- data_selected_list[["human"]] %>% filter(is_TF)
data_selected_list[["mouse"]]<- data_selected_list[["mouse"]] %>% filter(is_TF)

Let's also focus on gene pairs within a 10Kb window

In [9]:
sel_window_size = 10000
# add absolute strand distance and filter
data_selected_list <- lapply(data_selected_list, function(df) {
    df <- df %>% mutate(abs_strand_distance = abs(strand_distance)) %>%
        filter(abs_strand_distance <= sel_window_size)
    return(df)
})

In [10]:
# selected pairs after filtering
cat("Number of human lncRNA-TF pairs:", nrow(data_selected_list[["human"]]), "\n")
cat("Number of mouse lncRNA-TF pairs:", nrow(data_selected_list[["mouse"]]), "\n")

Number of human lncRNA-TF pairs: 55454 
Number of mouse lncRNA-TF pairs: 7263 


#### Annoate gene_ids to create pairs

Since in the current data frame we don't have Gene IDs, let's load the original ENSEMBL annotation to map them

In [11]:
# create a list to store ENSEMBL annotations 
ensembl_annotations_list <- list()

# Get the files within the ENSEMBL directory
ensembl_files <- list.files(ensembl_dir)

# Select the files related to the species in data_selected_list
for (species in names(data_selected_list)) {
    ensembl_file <- ensembl_files[str_detect(ensembl_files, species)]
    ensembl_data <- read.table(file.path(ensembl_dir, ensembl_file), sep = "\t", header = TRUE, 
                               stringsAsFactors = FALSE, quote = "", 
                               comment.char = "", fill = TRUE, row.names = NULL)
    ensembl_annotations_list[[species]] <- ensembl_data
}

head(ensembl_annotations_list[["human"]], 3)


,row.names,Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Gene.stable.ID,Transcript.stable.ID,Transcript.start..bp.,Transcript.end..bp.,Transcript.type,Gene.type,Gene.name,Gene.description,TSS,is_pcg,is_ncrna
,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>,<lgl>
1,1,1,11121,24894,1,ENSG00000290825,ENST00000456328,11850,14416,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11850,FALSE,TRUE
2,2,1,11121,24894,1,ENSG00000290825,ENST00000832823,14404,24894,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],14404,FALSE,TRUE
3,3,1,11121,24894,1,ENSG00000290825,ENST00000832824,11121,14413,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11121,FALSE,TRUE


In [12]:
# Transfer the ensembl ids to the selected data
for (species in names(data_selected_list)) {
    tmp_ensembl <- ensembl_annotations_list[[species]] %>%
                      select(Gene.stable.ID, Transcript.stable.ID) %>%
                      dplyr::rename(gene_id = Gene.stable.ID, transcript_id = Transcript.stable.ID)

    # transfer the gene ids for the lncRNAs
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        left_join(tmp_ensembl,by = c("ncRNA_id" = "transcript_id"))  %>%
        dplyr::rename(ncrna_gene_id = gene_id)
        
    # transfer the gene ids for the PCGs
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        left_join(tmp_ensembl,by = c("pcg_id" = "transcript_id"))  %>%
        dplyr::rename(pcg_gene_id = gene_id)
                    
}

In [13]:
head(data_selected_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>
1,11,ENST00000244906,61757671,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-2282,2282,NDT80_PhoG,TRUE,2282,ENSG00000124915,ENSG00000124920
2,17,ENST00000285274,18682228,ZNF286B,-1,lncRNA,ENST00000454745,FOXO3B,18682262,34,-34,Fork_head,TRUE,34,ENSG00000290687,ENSG00000240445
3,5,ENST00000297163,136134890,SMAD5-AS1,-1,lncRNA,ENST00000506223,SMAD5,136133823,-1067,1067,MH1,TRUE,1067,ENSG00000164621,ENSG00000113658


#### Selection of gene pairs

We would like to select unique gene pairs, for example, we expect duplicated distance relationships between lncRNA-TF transcripts.

Let's select the nearest pair as the representative lncRNA-TF pair

We have the issue of missing gene names. 
In this case let's assign them as unnamed. In any case, the main pair annotation will be formed using gene ids, which doesn't have this problem

In [19]:
# Assign missing lncRNA or TF gene names as unnamed 
for(species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        mutate(ncrna_gene_name = ifelse(ncrna_gene_name == "", "unnamed", ncrna_gene_name),
               pcg_gene_name = ifelse(pcg_gene_name == "", "unnamed", pcg_gene_name))
}

In [20]:
# create a gene_pair_id column using the ncrna_gene_name and pcg_gene_name
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        mutate(gene_pair_id = paste0(ncrna_gene_id, "_", pcg_gene_id))
}

# create a gene_name_pair_id column using the ncrna_gene_name and pcg_gene_name
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        mutate(gene_name_pair_id = paste0(ncrna_gene_name, "_", pcg_gene_name))
}

head(data_selected_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id,gene_pair_id,gene_name_pair_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>
1,11,ENST00000244906,61757671,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-2282,2282,NDT80_PhoG,TRUE,2282,ENSG00000124915,ENSG00000124920,ENSG00000124915_ENSG00000124920,MYRF-AS1_MYRF
2,17,ENST00000285274,18682228,ZNF286B,-1,lncRNA,ENST00000454745,FOXO3B,18682262,34,-34,Fork_head,TRUE,34,ENSG00000290687,ENSG00000240445,ENSG00000290687_ENSG00000240445,ZNF286B_FOXO3B
3,5,ENST00000297163,136134890,SMAD5-AS1,-1,lncRNA,ENST00000506223,SMAD5,136133823,-1067,1067,MH1,TRUE,1067,ENSG00000164621,ENSG00000113658,ENSG00000164621_ENSG00000113658,SMAD5-AS1_SMAD5


Now, let's get unique gene pair ids

In [22]:
# for each gene_pair_id, keep only the one with the smallest abs_strand_distance
unique_pairs_list <- lapply(data_selected_list, function(df) {
    df <- df %>%
        group_by(gene_pair_id) %>%
        slice_min(order_by = abs_strand_distance, n = 1, with_ties = FALSE) %>%
        ungroup()
    return(df)
})

In [23]:
cat("Number of human pairs before removing duplicates (gene_name_pair_id):\n", nrow(data_selected_list[["human"]]))
cat("\nNumber of human pairs after removing duplicates (gene_name_pair_id):\n", nrow(unique_pairs_list[["human"]]))
cat("\n\nNumber of mouse pairs before removing duplicates (gene_name_pair_id):\n", nrow(data_selected_list[["mouse"]]))
cat("\nNumber of mouse pairs after removing duplicates (gene_name_pair_id):\n", nrow(unique_pairs_list[["mouse"]]))

Number of human pairs before removing duplicates (gene_name_pair_id):
 55454
Number of human pairs after removing duplicates (gene_name_pair_id):
 1978

Number of mouse pairs before removing duplicates (gene_name_pair_id):
 7263
Number of mouse pairs after removing duplicates (gene_name_pair_id):
 1470

In [25]:
# Save the unique pairs data
for (species in names(unique_pairs_list)) {
    out_file <- file.path(out_dir, paste0("/Annotated_ncRNA_PCG_pairs/", species, "_unique_lncRNA_TF_pairs_", sel_window_size, "bp_", date, ".tsv"))
    write.table(unique_pairs_list[[species]], file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)
}

Several gene pairs have incomplete gene annotations, e.g., lncRNA or PCG gene name are missing. In this case, we have a technical issue since the TF gene family annotations are not so accurate, for example same PCG transcript with missing gene_name can be assigned to multiple families. Since the distance doesn't change, we don't have a clear way to assess the enrichment of a certain family in this case. Therefore, for the total number of counts per TF family, we will only used PCGs with known gene_name to avoid this issues

In [51]:
# Let's get the table of enriched TF families
# Let's keep the top 20 families and fof the rest, label them as "Other"
top_n = 20
table_tf_family <- unique_pairs_list[["human"]] %>% 
        filter(pcg_gene_name != "") %>% 
        group_by(Family) %>% tally() %>% 
        arrange(desc(n)) %>% 
        ungroup() %>%
        mutate(order = row_number()) %>% 
        mutate(Family = ifelse(order <= top_n, Family, "Others")) %>%
        group_by(Family) %>%
        summarise(n = sum(n), .groups = 'drop') %>%
        arrange(desc(n))

# Set Others in the end of the table
order <- table_tf_family$Family
order <- c(order[order != "Others"], "Others")
table_tf_family <- table_tf_family[match(order, table_tf_family$Family), ]
head(table_tf_family, 5)
tail(table_tf_family, 5)

# Save supplementary table of TF families
write.table(table_tf_family, file = file.path(out_dir, paste0("/Supplementary/Supplementary_Table_TF_families_counts_top", top_n, ".tsv")), 
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

Family,n
<chr>,<int>
zf-C2H2,554
Homeobox,275
bHLH,123
Fork_head,78
TF_bZIP,74


Family,n
<chr>,<int>
CUT,12
RXR-like,12
CSD,11
RFX,11
Others,276
